# Evaluate every metric

Calls the same library as `scripts/evaluate.py`, so the terminal and this notebook
cannot disagree.

| family | what | cost |
|---|---|---|
| `content` | LCF-WER + error breakdown, ICR@2, NRR | seconds when cached, ~3 s/clip when not |
| `signal` | SDR / SIR / SAR | ~1 min |
| `perceptual` | DNSMOS P.808 and P.835 | ~2.8 s per clip per system |

Systems: `floor` (unprocessed mixture), `estimate` (your model), `ceiling` (clean target).

**Paths need no `..` prefixes.** Relative paths resolve against the repo root,
which the library derives from its own file location — so it does not matter where
this notebook is opened from.

It does **not** render estimates — that is `scripts/make_estimates.py`, ~25 min.

## Setup

Walks up from the working directory to find the repo, so the import works whether
you launched Jupyter from the repo root or from `notebooks/`.

`%autoreload 2` means edits to `src/live_model_metric/` are picked up without a
kernel restart. **If you ever see an `ImportError` for a name that is clearly in
the file, the kernel is holding a cached module — restart it.**

In [ ]:
# autoreload picks up edits to src/live_model_metric/ WITHOUT restarting the
# kernel. Without it, Python caches the module in sys.modules and you get a
# stale version -- which looks like an ImportError for a symbol that plainly
# exists in the file.
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src" / "live_model_metric").is_dir())
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src.live_model_metric.evaluate import evaluate, REPO_ROOT
print("repo:", REPO_ROOT)

## 1. Quick check — content only, anchors only, 20 trials

`allow_new_transcripts=False` refuses to run the ASR, so this either returns in
seconds or tells you exactly what is missing. Use it to confirm the wiring.

In [ ]:
quick = evaluate(
    split="sir0_val",
    systems=("floor", "ceiling"),
    metrics=("content",),
    limit=20,
    allow_new_transcripts=False,
)
print(quick.table())

## 2. The full run

All three systems, all three families. ~15 minutes on 103 trials, almost all DNSMOS.

`estimate_directory` is relative to the repo root — no `..` needed.

In [ ]:
results = evaluate(
    split="sir0_val",
    condition="both",                 # the only condition with an interferer to remove
    estimate_directory="experiments/results/2026-09-01-est-sir0-5000",
    systems=("floor", "estimate", "ceiling"),
    metrics=("content", "signal", "perceptual"),
)
print(results.table())

### Cheaper variants

If memory is tight, DNSMOS is the expensive part and can be split out — an earlier
full run was OOM-killed at 232 MB free.

In [ ]:
# content + signal only: about a minute
fast = evaluate(
    split="sir0_val",
    estimate_directory="experiments/results/2026-09-01-est-sir0-5000",
    metrics=("content", "signal"),
)
print(fast.table())

## 3. As a dataframe

In [ ]:
results.frame()

In [ ]:
# headroom captured: how far the model moved from floor towards ceiling
for metric in ("lcf_wer", "icr_at_2"):
    floor, model, ceiling = (results.systems[s][metric]
                             for s in ("floor", "estimate", "ceiling"))
    print(f"{metric:<10} {100*(floor-model)/(floor-ceiling):5.1f} % of available headroom")

## 4. Save

Writes `results.json` (with provenance: commit, date, split, listener identity)
and `results.txt`. Path is relative to the repo root.

In [ ]:
results.save("experiments/results/notebook-evaluate-sir0_val")

## Caveats that travel with every number here

- **The listener is an offline ASR standing in for the judge.** Not live-model
  results. Until a judge is chosen `lcf_wer` is arithmetically identical to
  offline ASR word error rate.
- **`sir0_val` is the harder set** — symmetric loudness by construction, so its
  floor is worse than `eval_public`'s.
- **`nrr` is near-zero by construction** with an ASR, since a transcriber cannot
  decline. Read it as "not yet measurable", not as a result.
- **Ceilings are not perfect**: LCF-WER's is 5.8 %, DNSMOS `OVRL`'s is 3.43,
  because the reference is the *reverberant* target. Never quote a score without
  its ceiling.
- **Latency and RTF are not here** — they are properties of the model, not of a
  system row. Use `scripts/measure_rtf.py`.